# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：/Users/xuyifan/Desktop/ecommerce-user-analysis-seed/data/E Commerce Dataset.xlsx
项目输出目录：/Users/xuyifan/Desktop/ecommerce-user-analysis-seed/output/day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [2]:
# 1. 每条记录代表一名电商用户，数据粒度是“用户级”，不是订单级。
# 2. 项目的目标变量是 Churn，其中 1 表示用户流失，0 表示用户未流失。
# 3. CustomerID 只是用户唯一标识符。编号大小没有连续业务含义，
#    因此不应计算均值、相关系数，也不应作为普通连续特征直接参与分析。

record_meaning = "每条记录代表一名电商用户"
target_column = "Churn"
customer_id_note = "CustomerID仅用于唯一识别用户，编号大小不具有连续业务含义"

print("记录含义：", record_meaning)
print("目标变量：", target_column)
print("CustomerID说明：", customer_id_note)


记录含义： 每条记录代表一名电商用户
目标变量： Churn
CustomerID说明： CustomerID仅用于唯一识别用户，编号大小不具有连续业务含义


---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [3]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    report = pd.DataFrame({
        "数据类型": data.dtypes.astype(str),
        "缺失数量": data.isna().sum().astype(int),
        "缺失比例(%)": (data.isna().mean() * 100).round(2),
        "唯一值数量": data.nunique(dropna=True).astype(int),
    })
    report.index.name = "字段"
    return report.sort_values(
        ["缺失数量", "唯一值数量"],
        ascending=[False, True],
    )


# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)


,数据类型,缺失数量,缺失比例(%),唯一值数量
字段,,,,
DaySinceLastOrder,float64,307,5.45,22
OrderAmountHikeFromlastYear,float64,265,4.71,16
Tenure,float64,264,4.69,36
OrderCount,float64,258,4.58,16
CouponUsed,float64,256,4.55,17
HourSpendOnApp,float64,255,4.53,6
WarehouseToHome,float64,251,4.46,34
Churn,int64,0,0.00,2
Gender,object,0,0.00,2


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [4]:
# 项目初始审计
duplicate_rows_before = int(raw_df.duplicated().sum())
customer_id_duplicates = int(raw_df["CustomerID"].duplicated().sum())
churn_counts = raw_df["Churn"].value_counts(dropna=False).sort_index()
churn_rate = float(pd.to_numeric(raw_df["Churn"], errors="coerce").mean())

print("完全重复行数：", duplicate_rows_before)
print("CustomerID重复数量：", customer_id_duplicates)
print("\nChurn频数：")
print(churn_counts)
print(f"流失率：{churn_rate:.2%}")

for col in [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]:
    print(f"\n{col}频数：")
    print(raw_df[col].value_counts(dropna=False))


完全重复行数： 0
CustomerID重复数量： 0

Churn频数：
Churn
0    4682
1     948
Name: count, dtype: int64
流失率：16.84%

PreferredLoginDevice频数：
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode频数：
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat频数：
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [5]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [6]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # 深拷贝，确保原始数据只读
    cleaned_df = data.copy(deep=True)
    logs = []

    def add_log(step, rule, before_rows, after_rows, affected_rows):
        logs.append({
            "处理步骤": step,
            "处理规则": rule,
            "处理前记录数": int(before_rows),
            "处理后记录数": int(after_rows),
            "影响记录数": int(affected_rows),
        })

    # 1. 删除完全重复行
    before_rows = len(cleaned_df)
    duplicate_count = int(cleaned_df.duplicated().sum())
    cleaned_df = cleaned_df.drop_duplicates().copy()
    add_log(
        "完全重复记录处理",
        "删除所有字段完全相同的重复记录",
        before_rows,
        len(cleaned_df),
        duplicate_count,
    )

    # 2. 数值缺失值：使用全体样本中位数填补，不使用Churn分组
    for col in NUMERIC_MISSING_COLS:
        if col not in cleaned_df.columns:
            raise KeyError(f"缺少待清洗字段：{col}")

        before_rows = len(cleaned_df)
        cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors="coerce")
        missing_count = int(cleaned_df[col].isna().sum())
        median_value = cleaned_df[col].median()

        if missing_count > 0 and pd.isna(median_value):
            raise ValueError(f"{col}全部为空，无法使用中位数填补")

        cleaned_df[col] = cleaned_df[col].fillna(median_value)
        add_log(
            f"{col}缺失值处理",
            f"使用总体中位数{median_value:.2f}填补；不按Churn分组",
            before_rows,
            len(cleaned_df),
            missing_count,
        )

    # 3. 类别字段先清除首尾空格，再完成同义值标准化
    for col, mapping in CATEGORY_MAPPINGS.items():
        if col not in cleaned_df.columns:
            raise KeyError(f"缺少待清洗字段：{col}")

        before_rows = len(cleaned_df)
        # 保留真正的缺失值，不把NaN转换成字符串“nan”
        cleaned_df[col] = cleaned_df[col].astype("string").str.strip()

        for old_value, new_value in mapping.items():
            affected_count = int(cleaned_df[col].eq(old_value).sum())
            cleaned_df[col] = cleaned_df[col].replace(old_value, new_value)
            add_log(
                f"{col}类别标准化",
                f"{old_value} → {new_value}",
                before_rows,
                len(cleaned_df),
                affected_count,
            )

    # 4. 二元标签转换为整数类型
    for col in ["Churn", "Complain"]:
        if col not in cleaned_df.columns:
            raise KeyError(f"缺少待转换字段：{col}")

        before_rows = len(cleaned_df)
        numeric_series = pd.to_numeric(cleaned_df[col], errors="coerce")
        invalid_count = int(numeric_series.isna().sum())

        if invalid_count > 0:
            raise ValueError(f"{col}存在{invalid_count}个无法转换为整数的值")

        non_binary_count = int((~numeric_series.isin([0, 1])).sum())
        if non_binary_count > 0:
            raise ValueError(f"{col}存在{non_binary_count}个非0/1取值")

        changed_count = int((cleaned_df[col].astype(str) != numeric_series.astype("int64").astype(str)).sum())
        cleaned_df[col] = numeric_series.astype("int64")
        add_log(
            f"{col}类型转换",
            "转换为int64，并验证取值仅为0和1",
            before_rows,
            len(cleaned_df),
            max(changed_count, invalid_count, non_binary_count),
        )

    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log


### 任务 3：运行清洗函数并查看日志

In [7]:
# 执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
display(cleaned_df.head())

print("清洗前形状：", raw_df.shape)
print("清洗后形状：", cleaned_df.shape)


,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,完全重复记录处理,删除所有字段完全相同的重复记录,5630,5630,0
1,Tenure缺失值处理,使用总体中位数9.00填补；不按Churn分组,5630,5630,264
2,WarehouseToHome缺失值处理,使用总体中位数14.00填补；不按Churn分组,5630,5630,251
3,HourSpendOnApp缺失值处理,使用总体中位数3.00填补；不按Churn分组,5630,5630,255
4,OrderAmountHikeFromlastYear缺失值处理,使用总体中位数15.00填补；不按Churn分组,5630,5630,265
5,CouponUsed缺失值处理,使用总体中位数1.00填补；不按Churn分组,5630,5630,256
6,OrderCount缺失值处理,使用总体中位数2.00填补；不按Churn分组,5630,5630,258
7,DaySinceLastOrder缺失值处理,使用总体中位数3.00填补；不按Churn分组,5630,5630,307
8,PreferredLoginDevice类别标准化,Phone → Mobile Phone,5630,5630,1231
9,PreferredPaymentMode类别标准化,COD → Cash on Delivery,5630,5630,365


字段,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


清洗前形状： (5630, 20)
清洗后形状： (5630, 20)


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [8]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = pd.to_numeric(series, errors="coerce").dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_mask = (series < lower) | (series > upper)
    outlier_count = int(outlier_mask.sum())

    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": outlier_count,
        "候选异常值比例(%)": round(outlier_count / len(series) * 100, 2)
        if len(series) else 0.0,
    }


# 用户生命周期分层：右闭区间
tenure_bins = [-np.inf, 3, 6, 12, 24, np.inf]
tenure_labels = [
    "新用户(≤3个月)",
    "成长期(4-6个月)",
    "稳定期(7-12个月)",
    "忠诚期(13-24个月)",
    "长期用户(>24个月)",
]

cleaned_df["TenureGroup"] = pd.cut(
    cleaned_df["Tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    right=True,
    ordered=True,
)

# 移动端登录为1，其余设备为0
cleaned_df["IsMobileLogin"] = (
    cleaned_df["PreferredLoginDevice"]
    .eq("Mobile Phone")
    .astype("int64")
)

# IQR只用于生成候选异常值报告，不自动删除记录
outlier_fields = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_rows = []

for col in outlier_fields:
    summary = iqr_outlier_summary(cleaned_df[col])
    outlier_rows.append({
        "字段": col,
        **summary,
        "处理结论": "仅标记为候选异常值，保留原记录并提交业务复核",
    })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

print("\nTenureGroup频数：")
print(cleaned_df["TenureGroup"].value_counts(sort=False, dropna=False))
print("\nIsMobileLogin频数：")
print(cleaned_df["IsMobileLogin"].value_counts().sort_index())


,字段,Q1,Q3,IQR,下限,上限,候选异常值数量,候选异常值比例(%),处理结论
0,WarehouseToHome,9.00,20.00,11.00,-7.50,36.50,2,0.04,仅标记为候选异常值，保留原记录并提交业务复核
1,OrderCount,1.00,3.00,2.00,-2.00,6.00,703,12.49,仅标记为候选异常值，保留原记录并提交业务复核
2,CashbackAmount,145.77,196.39,50.62,69.84,272.33,438,7.78,仅标记为候选异常值，保留原记录并提交业务复核



TenureGroup频数：
TenureGroup
新用户(≤3个月)       1560
成长期(4-6个月)       590
稳定期(7-12个月)     1584
忠诚期(13-24个月)    1467
长期用户(>24个月)      429
Name: count, dtype: int64

IsMobileLogin频数：
IsMobileLogin
0    1634
1    3996
Name: count, dtype: int64


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [9]:
# 业务规则检查
business_rules = [
    {
        "规则": "使用时长不能小于0",
        "字段": "Tenure",
        "判断条件": "Tenure < 0",
        "不合规记录数": int((cleaned_df["Tenure"] < 0).sum()),
    },
    {
        "规则": "仓库距离不能小于0",
        "字段": "WarehouseToHome",
        "判断条件": "WarehouseToHome < 0",
        "不合规记录数": int((cleaned_df["WarehouseToHome"] < 0).sum()),
    },
    {
        "规则": "订单数必须大于0",
        "字段": "OrderCount",
        "判断条件": "OrderCount <= 0",
        "不合规记录数": int((cleaned_df["OrderCount"] <= 0).sum()),
    },
    {
        "规则": "返现金额不能小于0",
        "字段": "CashbackAmount",
        "判断条件": "CashbackAmount < 0",
        "不合规记录数": int((cleaned_df["CashbackAmount"] < 0).sum()),
    },
]

business_rule_report = pd.DataFrame(business_rules)
business_rule_report["处理结论"] = np.where(
    business_rule_report["不合规记录数"].eq(0),
    "未发现不合规记录，无需处理",
    "不自动删除或修改，保留原记录并提交业务复核",
)

display(business_rule_report)

total_invalid = int(business_rule_report["不合规记录数"].sum())
if total_invalid == 0:
    print("处理结论：四项业务规则均未发现不合规记录。")
else:
    print(
        f"处理结论：共发现{total_invalid}条次不合规记录；"
        "当前仅记录并保留，不在缺少业务依据时自动删除或修正。"
    )


,规则,字段,判断条件,不合规记录数,处理结论
0,使用时长不能小于0,Tenure,Tenure < 0,0,未发现不合规记录，无需处理
1,仓库距离不能小于0,WarehouseToHome,WarehouseToHome < 0,0,未发现不合规记录，无需处理
2,订单数必须大于0,OrderCount,OrderCount <= 0,0,未发现不合规记录，无需处理
3,返现金额不能小于0,CashbackAmount,CashbackAmount < 0,0,未发现不合规记录，无需处理


处理结论：四项业务规则均未发现不合规记录。


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [10]:
# 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 最终验收
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0,     "指定数值字段仍有缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].dropna().unique(),     "登录设备仍包含Phone"
assert "COD" not in cleaned_df["PreferredPaymentMode"].dropna().unique(),     "支付方式仍包含COD"
assert "CC" not in cleaned_df["PreferredPaymentMode"].dropna().unique(),     "支付方式仍包含CC"
assert "Mobile" not in cleaned_df["PreferedOrderCat"].dropna().unique(),     "偏好品类仍包含Mobile"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns),     "缺少数据转换字段"
assert cleaned_df.duplicated().sum() == 0, "清洗后仍有完全重复行"
assert set(cleaned_df["Churn"].unique()).issubset({0, 1}),     "Churn存在非0/1值"
assert set(cleaned_df["Complain"].unique()).issubset({0, 1}),     "Complain存在非0/1值"
assert cleaned_df["CustomerID"].notna().all(), "CustomerID存在缺失值"

print("清洗前缺失值总数：", int(raw_df.isna().sum().sum()))
print("清洗后缺失值总数：", int(cleaned_df.isna().sum().sum()))
display(quality_after)

# 导出全部交付物
output_files = {
    "清洗前质量报告": OUTPUT_DIR / "data_quality_before.csv",
    "清洗后质量报告": OUTPUT_DIR / "data_quality_after.csv",
    "清洗日志": OUTPUT_DIR / "cleaning_log.csv",
    "候选异常值报告": OUTPUT_DIR / "outlier_report.csv",
    "业务规则报告": OUTPUT_DIR / "business_rule_report.csv",
    "清洗后用户数据": OUTPUT_DIR / "ecommerce_customer_cleaned.csv",
}

quality_before.to_csv(
    output_files["清洗前质量报告"],
    index=True,
    encoding="utf-8-sig",
)
quality_after.to_csv(
    output_files["清洗后质量报告"],
    index=True,
    encoding="utf-8-sig",
)
cleaning_log.to_csv(
    output_files["清洗日志"],
    index=False,
    encoding="utf-8-sig",
)
outlier_report.to_csv(
    output_files["候选异常值报告"],
    index=False,
    encoding="utf-8-sig",
)
business_rule_report.to_csv(
    output_files["业务规则报告"],
    index=False,
    encoding="utf-8-sig",
)
cleaned_df.to_csv(
    output_files["清洗后用户数据"],
    index=False,
    encoding="utf-8-sig",
)

print("\n项目验收通过，已生成以下文件：")
for name, path in output_files.items():
    print(f"- {name}：{path.relative_to(PROJECT_ROOT)}")


清洗前缺失值总数： 1856
清洗后缺失值总数： 0


,数据类型,缺失数量,缺失比例(%),唯一值数量
字段,,,,
Churn,int64,0,0.00,2
PreferredLoginDevice,string,0,0.00,2
Gender,object,0,0.00,2
Complain,int64,0,0.00,2
IsMobileLogin,int64,0,0.00,2
CityTier,int64,0,0.00,3
MaritalStatus,object,0,0.00,3
PreferredPaymentMode,string,0,0.00,5
PreferedOrderCat,string,0,0.00,5



项目验收通过，已生成以下文件：
- 清洗前质量报告：output/day04_project/data_quality_before.csv
- 清洗后质量报告：output/day04_project/data_quality_after.csv
- 清洗日志：output/day04_project/cleaning_log.csv
- 候选异常值报告：output/day04_project/outlier_report.csv
- 业务规则报告：output/day04_project/business_rule_report.csv
- 清洗后用户数据：output/day04_project/ecommerce_customer_cleaned.csv


## 项目复盘

本项目发现数值字段缺失、同义类别命名不一致，并检查了重复记录、候选异常值和业务不合规值。缺失值采用总体中位数填补，不使用Churn分组；类别按映射统一；IQR仅用于标记，不自动删除。清洗后核心字段完整、类别一致，并新增TenureGroup和IsMobileLogin，可作为第五天分析输入。异常边界及不合规记录是否修正仍需业务人员确认。


## GitHub提交检查

- [x] Notebook代码已补全，可重启内核并从头运行；
- [x] 运行后`output/day04_project/`会生成清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [x] 原始Excel只读，不会被覆盖；
- [x] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 在本机使用真实数据运行全部单元后，提交并推送到个人GitHub仓库。
